# Task 04: Supervised Classification Modeling & Tuning
**Problem:** Predict customer churn (binary classification) using the churn training dataset from Tasks 02 and 03.

**Plan:** (1) load data and split before any transform, (2) reuse the leak-free preprocessing pipeline, (3) train 5 model architectures with stratified K-fold CV, (4) tune hyperparameters, (5) evaluate on the held-out test set (Precision, Recall, F1, ROC-AUC, Confusion Matrix), (6) pick a champion and serialize it.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay)
try:
    from xgboost import XGBClassifier; HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    from lightgbm import LGBMClassifier; HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
print("XGBoost:", HAS_XGB, "| LightGBM:", HAS_LGBM)

## 1. Load data and split (before any transformation)

In [ ]:
DATA_PATH = "churn_train.csv"   # <-- change to your churn CSV filename
TARGET    = "Churn"             # <-- change to your target column name

df = pd.read_csv(DATA_PATH)
print(df.shape); display(df.head())

y = df[TARGET]
if y.dtype == object:
    y = y.astype(str).str.strip().str.lower().map({"yes":1,"no":0,"1":1,"0":0,"true":1,"false":0})
X = df.drop(columns=[TARGET])
# drop ID-like columns (no predictive value, leakage risk)
id_cols = [c for c in X.columns if "id" == c.lower() or c.lower().endswith("id") and X[c].nunique()==len(X)]
X = X.drop(columns=id_cols)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape, "| churn rate:", round(y.mean(),3))

## 2. Preprocessing pipeline (fit on training folds only, so there is no leakage)

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3. Models and hyperparameter grids (5 distinct architectures)

In [ ]:
spw = (y_train==0).sum()/(y_train==1).sum()   # class imbalance ratio

models = {
 "Logistic Regression": (LogisticRegression(max_iter=2000, class_weight="balanced"),
     {"clf__C":[0.01,0.1,1,10], "clf__penalty":["l2"]}),
 "Random Forest": (RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
     {"clf__n_estimators":[200,400], "clf__max_depth":[None,10,20], "clf__min_samples_leaf":[1,3,5]}),
 "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
     {"clf__n_estimators":[100,200], "clf__learning_rate":[0.05,0.1], "clf__max_depth":[2,3]}),
 "KNN": (KNeighborsClassifier(),
     {"clf__n_neighbors":[5,11,21], "clf__weights":["uniform","distance"]}),
}
if HAS_XGB:
    models["XGBoost"] = (XGBClassifier(eval_metric="logloss", scale_pos_weight=spw, random_state=42, n_jobs=-1),
        {"clf__n_estimators":[200,400], "clf__max_depth":[3,5], "clf__learning_rate":[0.05,0.1]})
if HAS_LGBM:
    models["LightGBM"] = (LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1),
        {"clf__n_estimators":[200,400], "clf__num_leaves":[15,31], "clf__learning_rate":[0.05,0.1]})
print(list(models))

## 4. Hyperparameter optimization with stratified 5-fold CV (scoring: ROC-AUC)

In [ ]:
tuned, cv_rows = {}, []
for name,(est,grid) in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", est)])
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc", n_jobs=-1)
    gs.fit(X_train, y_train)
    tuned[name] = gs.best_estimator_
    cv_rows.append({"Model":name, "CV ROC-AUC":round(gs.best_score_,4), "Best Params":gs.best_params_})
    print(f"{name}: CV AUC={gs.best_score_:.4f} | {gs.best_params_}")
cv_df = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC", ascending=False); display(cv_df)

## 5. Evaluation on the held-out test set

In [ ]:
rows, probs = [], {}
for name, m in tuned.items():
    p = m.predict_proba(X_test)[:,1]; pred = (p>=0.5).astype(int); probs[name]=p
    rows.append({"Model":name,
                 "Precision":precision_score(y_test,pred), "Recall":recall_score(y_test,pred),
                 "F1":f1_score(y_test,pred), "ROC-AUC":roc_auc_score(y_test,p)})
results = pd.DataFrame(rows).round(4).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
display(results)

In [ ]:
plt.figure(figsize=(7,6))
for name,p in probs.items():
    fpr,tpr,_ = roc_curve(y_test,p); plt.plot(fpr,tpr,label=f"{name} (AUC={roc_auc_score(y_test,p):.3f})")
plt.plot([0,1],[0,1],"k--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves, all models"); plt.legend(); plt.grid(alpha=.3); plt.savefig("roc_curves.png", dpi=150); plt.show()

In [ ]:
n = len(tuned); cols = 3; rws = int(np.ceil(n/cols))
fig, axes = plt.subplots(rws, cols, figsize=(5*cols, 4.2*rws)); axes = np.array(axes).ravel()
for ax,(name,m) in zip(axes, tuned.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, m.predict(X_test))).plot(ax=ax, colorbar=False); ax.set_title(name)
for ax in axes[n:]: ax.axis("off")
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

## 6. Champion model selection and serialization

In [ ]:
# Champion = best validation (CV) ROC-AUC; tie-break by test F1 (churn needs a recall/precision balance)
champion_name = cv_df.iloc[0]["Model"]
champion = tuned[champion_name]
print("Champion:", champion_name)
display(results[results.Model==champion_name])
joblib.dump(champion, "champion_churn_model.joblib")
print("Saved champion_churn_model.joblib")

## 7. Conclusion
- **Champion model:** *(fill in from output above)*, selected on stratified 5-fold CV ROC-AUC.
- **Why:** *(compare its Recall/F1 against the others; for churn, missing a churner (false negative) is usually costlier than a false alarm)*.
- **Limitations and next steps:** threshold tuning, probability calibration, SHAP explanations, and monitoring for drift (links back to the risk register in Task 02).